# Experiment 3: Small SFT for Mechanism→Application (B→C) Internalization

**Goal:** Fine-tune the model with a small supervised dataset so it *internalizes* the bridge-execution behavior pattern — reliably stating a concise mechanism **B** before producing a constraint-satisfying solution **C**.

**Key design fix vs. original:** Training uses a **plain system prompt** (not a mechanism-priming one). This forces the model to learn mechanism-first reasoning as a *default* mode, not a conditional response to a specific instruction.

**Task:** Set **TASK_TYPE** in the data cell to **"ps"** (Problem Solving; `abcd_aiden_150_fixed.json`) or **"aut"** (Alternative Uses Task; `abcd_aut.json`).

## Protocol

1. **Build SFT pairs** from the ABCD dataset:
   - **Input:** plain system prompt + problem A
   - **Target:** `Mechanism: {B.type} — {B.rule}\n\n{C}`
2. **Single system prompt:** All training examples use the same plain system prompt.
3. **Split:** rest train / 35 eval (held-out, seeded random).
4. **LoRA fine-tuning** (rank 32, α=64) on attention + MLP projections.
5. **Short training:** 5–8 epochs at lr=2e-5.
6. **Evaluate:**
   - Pre-SFT vs Post-SFT on held-out eval items.
   - Does the model produce mechanism→solution **without** a special system prompt?
   - Generalization on novel problems not in the dataset.


In [ ]:
# DLP package (run from DLP/ or notebooks/)
import sys
from pathlib import Path
_root = Path.cwd()
for _ in range(10):
    if (_root / "dlp").is_dir():
        sys.path.insert(0, str(_root))
        break
    _root = _root.parent
from dlp.models import HFLoader
QwenModelLoader = HFLoader  # alias for rest of notebook


In [ ]:
"""Cell 1: Imports and QwenModelLoader."""

from __future__ import annotations

import json
import os
import re
import subprocess
import sys
from copy import deepcopy
from pathlib import Path
from typing import Any

# Prevent nvcc lookup: stub CUDA_HOME so code that expects CUDA_HOME/bin/nvcc does not fail.
os.environ.setdefault("BNB_CUDA_VERSION", "121")
import tempfile as _tempfile
_fake_cuda = os.path.join(_tempfile.gettempdir(), "fake_cuda_nvcc_stub")
_fake_bin = os.path.join(_fake_cuda, "bin")
os.makedirs(_fake_bin, exist_ok=True)
_nvcc = os.path.join(_fake_bin, "nvcc")
with open(_nvcc, "w") as _f:
    _f.write('#!/bin/sh\n')
    _f.write('V="Cuda compilation tools, release 12.1, V12.1.0"\n')
    _f.write('case "$1" in -V) echo "$V" ;; --version) echo "$V" ;; esac\n')
    _f.write('exit 0\n')
os.chmod(_nvcc, 0o755)
os.environ["CUDA_HOME"] = _fake_cuda

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    import peft  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "peft>=0.10.0"])


# QwenModelLoader from dlp.models (see previous cell)
# Verify nvcc stub (DeepSpeed uses -V, not --version)
for _arg in ("-V", "--version"):
    _nvcc_out = subprocess.run([_nvcc, _arg], capture_output=True, text=True)
    _has_release = "release" in (_nvcc_out.stdout or "") or "release" in (_nvcc_out.stderr or "")
    assert _has_release, f"nvcc stub {_arg} should print 'release'; got: " + repr(_nvcc_out.stdout + _nvcc_out.stderr)
print("nvcc stub OK (-V and --version); CUDA_HOME=", os.environ.get("CUDA_HOME", ""))
print("Imports OK.")


In [ ]:
"""Cell 2: Load model."""

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

loader = QwenModelLoader(MODEL_NAME)
loader.load()

model = loader.model
tokenizer = loader.tokenizer

n_layers = len(model.model.layers)
d_model = model.config.hidden_size
print(f"Model: {MODEL_NAME}")
print(f"Layers: {n_layers}, hidden_size: {d_model}")
print(f"Device: {next(model.parameters()).device}")

## 1. Build the SFT Dataset

Each training example teaches the model to produce **mechanism-first reasoning**:

- **System:** A minimal prompt that says "identify the mechanism, then apply it" (no elaborate decomposition instructions).
- **User:** The AUT task prompt A (object + constraints).
- **Assistant (target):** `Mechanism: {B.rule}\n\n{C solutions}`

The system prompt is intentionally *simple* — we want the model to internalize the B→C pattern even without detailed instructions. During evaluation, we’ll test with an even simpler system prompt (or none at all) to see if the pattern has been truly internalized.

**Train/eval split:** We hold out 35 items (seeded random) for evaluation.

In [ ]:
"""Cell 4: Load ABCD data and build SFT training/eval examples.

Set TASK_TYPE to "ps" (Problem Solving) or "aut" (Alternative Uses Task).
Training uses a plain system prompt so the model internalizes mechanism-first
reasoning as a default mode.
"""

import random as _rng

# ============================================================================
# Task type: "ps" or "aut"
# ============================================================================
  # "ps" = Problem Solving (abcd_aiden), "aut" = Alternative Uses (abcd_aut)
from enum import Enum
class TaskType(Enum):
    AUT = "aut"
    PS = "ps"


TASK_TYPE = TaskType.AUT  # <── change to "aut" for Alternative Uses Task   
DATASET_PATHS = {
    TaskType.AUT: "dataset/abcd_aut.json",
    TaskType.PS: "dataset/abcd_aiden.json",
}
DATASET_PATH = DATASET_PATHS[TASK_TYPE]
with open(DATASET_PATH) as f:
    abcd_data = json.load(f)

print(f"TASK_TYPE: {TASK_TYPE.value!r}, Loaded {len(abcd_data)} ABCD items from {DATASET_PATH}")

# ============================================================================
# Output directory: by task name and timestamp (for LLM-as-judge)
# ============================================================================
from datetime import datetime
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("results/sft_runs") / f"{TASK_TYPE.value}_{RUN_TIMESTAMP}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# ============================================================================
# Format helpers (PS vs AUT)
# ============================================================================

def format_user_prompt_ps(item: dict) -> str:
    """Build the user prompt for a PS item. Asks for mechanism first, then solution (like AUT)."""
    problem = item["A"]
    return (
        f"Problem Solving Task\n\n"
        f"Problem: {problem}\n\n"
        f"Give one creative, non-obvious solution. State the mechanism in one line (Mechanism: <type> — <rule>), then your solution in a short paragraph."
    )


def format_target_ps(item: dict) -> str:
    """Build the assistant target: Mechanism line + Solution block (same two-part structure as AUT)."""
    b, c = item["B"], item["C"]
    return f"Mechanism: {b['type']} — {b['rule']}\n\nSolution:\n\n{c}"


def format_user_prompt_aut(item: dict) -> str:
    """Build the user prompt for an AUT item (object + request for 8 uses)."""
    task = item["A"]
    return (
        f"Alternative Uses Task\n\n"
        f"{task}\n\n"
        f"Give 8 unconventional uses. List them clearly."
    )


# Baseline eval: ask only for a creative solution (no "state the mechanism" instruction).
# Fairer comparison — we don't require a format we didn't teach the baseline.
def format_user_prompt_baseline_ps(item: dict) -> str:
    """User prompt for baseline (PS): creative solution only, no mechanism format."""
    problem = item["A"]
    return (
        f"Problem Solving Task\n\n"
        f"Problem: {problem}\n\n"
        f"Give one creative, non-obvious solution in a short paragraph."
    )


def format_user_prompt_baseline_aut(item: dict) -> str:
    """User prompt for baseline (AUT): same as standard — we don't ask for mechanism."""
    return format_user_prompt_aut(item)


def format_target_aut(item: dict) -> str:
    """Build the assistant target: Mechanism line + list of 8 uses (C is a list)."""
    b, c = item["B"], item["C"]
    uses = c if isinstance(c, list) else [c]
    uses_text = "\n".join(f"- {u}" for u in uses)
    return (
    f"Mechanism: {b['type']} — {b['rule']}\n"
    f"Note: Use this mechanism as a transferable bridge to generate non-obvious uses.\n"
    f"{uses_text}"
)
    # return f"Mechanism: {b['type']} — {b['rule']}\nBridge: apply mechanism to generate non-obvious uses.\n{uses_text}"

def format_target_aut(item: dict) -> str:
    """
    Target structure that forces real mechanism execution:
    Mechanism -> Bridge (3 derived constraints: knobs / enables / excludes) -> 8 uses.
    Uses are expected to satisfy the bridge constraints and avoid generic templates unless forced.
    """
    b, c = item["B"], item["C"]
    uses = c if isinstance(c, list) else [c]
    uses_text = "\n".join(f"- {u}" for u in uses)

    # Derive compact bridge constraints from B fields (when present)
    rule = (b.get("rule") or "").strip().rstrip(".")
    unlocks = (b.get("unlocks") or "").strip().rstrip(".")
    justification = (b.get("justification") or "").strip().rstrip(".")

    # "Knobs" = controllable levers implied by the mechanism (fallback heuristic)
    # Keep it short and generic; you can hand-tune later.
    knobs = []
    for k in ["tension", "friction", "pressure", "flow", "angle", "distance",
              "compression", "porosity", "seal", "headspace", "beam", "alignment",
              "stiffness", "damping", "fold", "geometry", "nozzle", "vent"]:
        if k in rule.lower():
            knobs.append(k)
    knobs_txt = ", ".join(knobs[:4]) if knobs else "one controllable property from the mechanism"

    # "Excludes" = safe guardrail: avoid generic office/workshop templates unless mechanism forces them
    excludes = "avoid generic templates unless mechanism forces them"

    return (
        f"Mechanism: {b['type']} — {rule}.\n"
        f"Bridge (use as transferable constraints to generate non-obvious uses):\n"
        f"- Knobs: {knobs_txt}\n"
        f"- Enables: {unlocks or 'a new capability implied by the mechanism'}\n"
        f"- Excludes: {excludes}\n\n"
        f"Uses:\n{uses_text}"
    )
# Single entry points used by training and eval (dispatch on TASK_TYPE)
format_user_prompt = format_user_prompt_ps if TASK_TYPE.value == "ps" else format_user_prompt_aut
format_user_prompt_baseline = (
    format_user_prompt_baseline_ps if TASK_TYPE.value == "ps" else format_user_prompt_baseline_aut
)
format_target = format_target_ps if TASK_TYPE.value == "ps" else format_target_aut

# Reversible experiment: run baseline + postsft with type-3 prompt: task (a) + return exactly 8, one per line with '-'
EVAL_PROMPT_TYPE = "type3_minimal"  # "sft_style" (default) or "type3_minimal"
TRAIN_PROMPT_TYPE = "type3_minimal"  # "sft_style" or "type3_minimal" — set to type3 to align training with type-3 eval

def format_user_prompt_type3(item: dict) -> str:
    """Type-3 eval prompt: task (a), return exactly 8, one per line with '-'."""
    task = item["A"]
    return f"{task}\nReturn exactly 8 unconventional uses.\nFormat: one use per line, starting with \"-\"."

def format_user_prompt_eval_baseline(item: dict) -> str:
    """Prompt for baseline eval: PS always canonical; AUT uses type3 if set, else baseline."""
    if TASK_TYPE.value == "ps":
        return format_user_prompt_baseline(item)
    if EVAL_PROMPT_TYPE == "type3_minimal":
        return format_user_prompt_type3(item)
    return format_user_prompt_baseline(item)

def format_user_prompt_eval_postsft(item: dict) -> str:
    """Prompt for postsft eval: PS always canonical; AUT uses type3 if set, else full SFT style."""
    if TASK_TYPE.value == "ps":
        return format_user_prompt_baseline(item)
    if EVAL_PROMPT_TYPE == "type3_minimal":
        return format_user_prompt_type3(item)
    return format_user_prompt(item)


# Training prompt: use type3 for AUT when TRAIN_PROMPT_TYPE == "type3_minimal", else SFT style
def format_user_prompt_for_training(item: dict) -> str:
    """Prompt for training: respects TRAIN_PROMPT_TYPE (type3_minimal for AUT only)."""
    if TASK_TYPE.value != "ps" and TRAIN_PROMPT_TYPE == "type3_minimal":
        return format_user_prompt_type3(item)
    return format_user_prompt(item)


def build_sft_messages(item: dict) -> list[dict[str, str]]:
    """Build a full chat-format training example (always with plain system prompt)."""
    return [
        {"role": "system", "content": ""},
        {"role": "user", "content": format_user_prompt_for_training(item)},
        {"role": "assistant", "content": format_target(item)},
    ]


# ============================================================================
# Train/eval split (seeded random, same as bridge_steering)
# ============================================================================

EVAL_SEED = 7
N_EVAL = 40
_eval_idxs = set(_rng.Random(EVAL_SEED).sample(range(len(abcd_data)), min(N_EVAL, len(abcd_data))))

train_items = [item for i, item in enumerate(abcd_data) if i not in _eval_idxs]
eval_items = [item for i, item in enumerate(abcd_data) if i in _eval_idxs]

print(f"Train: {len(train_items)} items, Eval: {len(eval_items)} items")
print(f"Eval indices (seed={EVAL_SEED}): {sorted(_eval_idxs)}")
print(f"Eval IDs: {[item['id'] for item in eval_items]}")

# ============================================================================
# Build training messages (plain system prompt for all)
# ============================================================================
train_messages = [build_sft_messages(item) for item in train_items]
print(f"Training examples: {len(train_messages)} (all with system prompt)")

# Preview one example
print("\n" + "="*80)
print("Example SFT training conversation:")
print("="*80)
for msg in train_messages[0]:
    print(f"\n[{msg['role'].upper()}]")
    print(msg["content"][:400])


In [ ]:
"""Cell 5: Tokenize SFT examples and build a HF Dataset.

We tokenize the full conversation (system + user + assistant) using the chat
template. Labels are set to -100 for all prompt tokens so the loss is computed
only on the assistant's response (standard SFT practice).
"""

from datasets import Dataset


def tokenize_sft_example(
    messages: list[dict[str, str]],
    tokenizer: AutoTokenizer,
    max_length: int = 1024,
) -> dict[str, list[int]]:
    """Tokenize a chat conversation for SFT with masked prompt labels.

    The prompt portion (system + user + generation prompt tokens) gets label=-100.
    Only the assistant completion tokens contribute to the loss.
    """
    # Full conversation text (system + user + assistant)
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    full_ids = tokenizer(full_text, truncation=True, max_length=max_length)["input_ids"]

    # Prompt-only text (system + user, with generation prompt appended)
    prompt_msgs = [m for m in messages if m["role"] != "assistant"]
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs, tokenize=False, add_generation_prompt=True,
    )
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=max_length)["input_ids"]
    prompt_len = len(prompt_ids)

    # Labels: -100 for prompt tokens, actual ids for completion tokens
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    # Pad labels to match full_ids length (should already match, but be safe)
    if len(labels) < len(full_ids):
        labels += [-100] * (len(full_ids) - len(labels))
    labels = labels[:len(full_ids)]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


# Tokenize all training examples
tokenized_examples = [tokenize_sft_example(msgs, tokenizer) for msgs in train_messages]

# Verify masking
ex = tokenized_examples[0]
n_prompt = sum(1 for l in ex["labels"] if l == -100)
n_completion = sum(1 for l in ex["labels"] if l != -100)
print(f"Example 0: {len(ex['input_ids'])} total tokens, "
      f"{n_prompt} prompt (masked), {n_completion} completion (trained)")

# Build HF Dataset
train_dataset = Dataset.from_dict({
    "input_ids": [ex["input_ids"] for ex in tokenized_examples],
    "attention_mask": [ex["attention_mask"] for ex in tokenized_examples],
    "labels": [ex["labels"] for ex in tokenized_examples],
})

print(f"\nTraining dataset: {len(train_dataset)} examples")
print(f"Token lengths: {[len(ex['input_ids']) for ex in tokenized_examples]}")

## 2. Baseline Evaluation (Pre-SFT)

Before fine-tuning, we evaluate the base model on the held-out eval items. For a **fair comparison**, we ask the baseline only for a **creative solution** (we do *not* ask it to "state the mechanism" or use any format we didn't train it on). Post-SFT eval uses the full prompt (mechanism + solution). We still report whether the baseline *spontaneously* produces a mechanism-like line.

In [ ]:
"""Cell 7: Baseline generation on eval items (pre-SFT)."""

MAX_NEW_TOKENS = 384


def generate_eval_outputs(
    loader: QwenModelLoader,
    eval_items: list[dict],
    system_prompt: str | None,
    tag: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    user_prompt_fn=None,
) -> list[dict]:
    """Generate outputs for eval items. user_prompt_fn: if None, use format_user_prompt; use format_user_prompt_baseline for baseline (creative solution only)."""
    if user_prompt_fn is None:
        user_prompt_fn = format_user_prompt
    results = []
    for eval_idx, item in tqdm(enumerate(eval_items), total=len(eval_items), desc=f"Eval ({tag})"):
        user_prompt = user_prompt_fn(item)
        messages: list[dict[str, str]] = []
        if system_prompt is not None:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})

        reply = loader.generate(
            messages, max_new_tokens=max_new_tokens, do_sample=False
        )
        results.append({
            "eval_idx": eval_idx,
            "id": item["id"],
            "tag": tag,
            "system": system_prompt or "(none)",
            "user_prompt": user_prompt,
            "reply": reply,
            "reply_len_words": len(reply.split()),
            "has_mechanism_line": bool(re.search(r"(?i)mechanism\s*:", reply)),
            "ground_truth_B": item["B"]["rule"],
            "ground_truth_C": item["C"],
        })
    return results


# Baseline: use EVAL_PROMPT_TYPE (sft_style = default; type3_minimal = task-only, reversible experiment)
all_baseline = generate_eval_outputs(
    loader, eval_items, "", tag="baseline",
    user_prompt_fn=format_user_prompt_eval_baseline,
)
baseline_df = pd.DataFrame(all_baseline)

print(f"\nBaseline results: {len(baseline_df)} outputs")
print("(Baseline prompt: creative solution only, no mechanism format requested)")
print("\nMechanism line detection (spontaneous, pre-SFT):")
print(baseline_df["has_mechanism_line"].mean())

In [ ]:
"""Cell 8: Display baseline outputs."""

for item in eval_items:
    print("=" * 90)
    print(f"ID: {item['id']}  |  Object: {item['A'][:60]}...")
    print(f"Ground-truth mechanism: {item['B']['rule']}")
    print("-" * 90)
    for row in all_baseline:
        if row["id"] == item["id"]:
            tag = row["tag"]
            reply = row["reply"][:400] + ("..." if len(row["reply"]) > 400 else "")
            has_b = "✓ mechanism" if row["has_mechanism_line"] else "✗ no mechanism"
            print(f"\n  [{tag}] ({has_b})")
            print(f"  {reply}")
    print()

## 3. LoRA SFT Setup

We use **LoRA** (Low-Rank Adaptation) for parameter-efficient fine-tuning:
- **Rank:** 16 (small enough to avoid overfitting on 16 examples)
- **Alpha:** 32 (effective scaling = α/r = 2)
- **Target modules:** attention projections (q, k, v, o) — these control how the model attends and routes information
- **Dropout:** 0.05 (light regularization)

Training is deliberately short: we want the model to learn the *output pattern* (mechanism-first), not memorize specific mechanisms.

In [ ]:
"""Cell 10: Setup LoRA and prepare model for training."""

from peft import LoraConfig, get_peft_model, TaskType

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)
# lora_config = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=16,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
#     bias="none",
# )
# Wrap model with LoRA adapters
model.enable_input_require_grads()  # required for gradient checkpointing
peft_model = get_peft_model(model, lora_config)

trainable, total = peft_model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"LoRA rank: {lora_config.r}, alpha: {lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")

In [ ]:
"""Cell 11: Configure Trainer and run SFT.

Training strategy for a tiny dataset (16 examples):
  - 3-5 epochs: enough to learn the format, not enough to memorize content
  - Low lr (2e-5): stable convergence
  - No warmup: dataset is too small to benefit
  - Gradient accumulation = 4 with batch_size=1: effective batch = 4
  - bf16 mixed precision: faster, matches model's native dtype
"""

import os
import tempfile
# Avoid nvcc lookup: create a stub so code that expects CUDA_HOME/bin/nvcc does not fail
os.environ.setdefault("BNB_CUDA_VERSION", "121")
_fake_cuda = os.path.join(tempfile.gettempdir(), "fake_cuda_nvcc_stub")
_fake_bin = os.path.join(_fake_cuda, "bin")
os.makedirs(_fake_bin, exist_ok=True)
_nvcc = os.path.join(_fake_bin, "nvcc")
with open(_nvcc, "w") as _f:
    _f.write('#!/bin/sh\n')
    _f.write('V="Cuda compilation tools, release 12.1, V12.1.0"\n')
    _f.write('case "$1" in -V) echo "$V" ;; --version) echo "$V" ;; esac\n')
    _f.write('exit 0\n')
os.chmod(_nvcc, 0o755)
os.environ["CUDA_HOME"] = _fake_cuda
try:
    import torch.utils.cpp_extension as _ce
    _ce.CUDA_HOME = _fake_cuda
except Exception:
    pass

from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# OUTPUT_DIR set in data cell (task + timestamp); ensure it exists
if "OUTPUT_DIR" not in dir():
    from datetime import datetime
    OUTPUT_DIR = Path("results/sft_runs") / f"aut_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# For stronger mechanism internalization / novelty: try 7–8 epochs or lr 1.5e-5
NUM_TRAIN_EPOCHS = 5
LEARNING_RATE = 1e-5

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=0,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none",
    seed=42,
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer configured.")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Steps per epoch: {len(train_dataset) // training_args.per_device_train_batch_size}")
print(f"  Total optimization steps: ~{(len(train_dataset) * int(training_args.num_train_epochs)) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

In [ ]:
"""Cell 12: Train!"""

train_result = trainer.train()

print(f"\nTraining complete.")
print(f"  Total steps: {train_result.global_step}")
print(f"  Final loss: {train_result.training_loss:.4f}")

# Save the LoRA adapter
adapter_path = OUTPUT_DIR / "lora_adapter"
peft_model.save_pretrained(str(adapter_path))
print(f"  LoRA adapter saved to: {adapter_path}")

In [ ]:
"""Cell 13: Plot training loss curve."""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if "trainer" not in dir() or trainer is None:
    print("No trainer in scope. Run the 'Train!' cell first.")
else:
    log_history = getattr(trainer.state, "log_history", [])
    train_losses = [
        (entry["step"], entry["loss"])
        for entry in log_history
        if entry.get("loss") is not None
    ]

    if train_losses:
        steps, losses = zip(*train_losses)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(steps, losses, "o-", markersize=3, color="tab:blue")
        ax.set_xlabel("Step")
        ax.set_ylabel("Training Loss")
        ax.set_title("SFT Training Loss (B\u2192C Internalization)")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        out_path = OUTPUT_DIR / "training_loss.png" if "OUTPUT_DIR" in dir() else Path("training_loss.png")
        plt.savefig(str(out_path), dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Loss curve saved to {out_path}")
    else:
        print("No training loss entries found in log history.")

## 4. Post-SFT Evaluation

Now we test whether the fine-tuned model has internalized the B→C pattern.

**Key question:** Does the model now produce `Mechanism: ...` followed by mechanism-informed solutions, even when given the **plain** system prompt or **no** system prompt at all?

We evaluate on:
1. **Held-out eval items** (objects seen in the dataset but not in training).
2. **Novel objects** (completely unseen) to test generalization.

In [ ]:
"""Cell 15: Post-SFT evaluation on held-out items.

We re-use the same generate_eval_outputs function but now the model has
LoRA adapters active, so generation passes through the fine-tuned weights.
"""
loader._model = peft_model  # make loader.generate() use the LoRA-wrapped model
# peft_model.eval()
peft_model.eval()

# Post-SFT with plain system prompt; same EVAL_PROMPT_TYPE as baseline (type3 = task-only)
all_postsft = generate_eval_outputs(
    loader, eval_items, "", tag="postsft",
    user_prompt_fn=format_user_prompt_eval_postsft,
)
postsft_df = pd.DataFrame(all_postsft)

print(f"\nPost-SFT results: {len(postsft_df)} outputs")
print("\nMechanism line detection (post-SFT):")
print(postsft_df["has_mechanism_line"].mean())
print("\nvs. Baseline (pre-SFT):")
print(baseline_df["has_mechanism_line"].mean())

In [ ]:
"""Cell 16: Side-by-side comparison: Pre-SFT vs Post-SFT."""

from IPython.display import display, HTML
import html as _html

_TRUNC = 60000

_card = (
    "border:2px solid #555; border-radius:8px; margin:20px 0; padding:0; "
    "font-family:system-ui,sans-serif; background:#1e1e1e; color:#e0e0e0;"
)
_hdr = (
    "background:#2d2d2d; padding:12px 16px; border-bottom:2px solid #555; "
    "border-radius:6px 6px 0 0;"
)
_row = (
    "border-bottom:1px solid #444; padding:12px 16px; "
    "white-space:pre-wrap; word-break:break-word; line-height:1.5; font-size:13px;"
)
_tag_td = (
    "vertical-align:top; width:140px; padding:12px 12px 12px 16px; "
    "border-right:2px solid #444; background:#252525; font-weight:bold; font-size:13px;"
)


def _color_tag(tag: str) -> str:
    if "baseline" in tag:
        return "color:#ff8888;"
    return "color:#88ff88;"


all_results = all_baseline + all_postsft

parts = []
for eval_idx, item in enumerate(eval_items):
    item_id = item["id"]
    gt_b = _html.escape(item["B"]["rule"])

    rows = []
    for r in all_results:
        if r.get("eval_idx") != eval_idx:
            continue
        tag = r["tag"]
        reply = r["reply"]
        short = _html.escape(reply[:_TRUNC]) + ("\u2026" if len(reply) > _TRUNC else "")
        badge = (
            '<span style="color:#44ff44;">\u2713 mechanism</span>'
            if r["has_mechanism_line"]
            else '<span style="color:#ff4444;">\u2717 no mechanism</span>'
        )
        rows.append(
            f'<tr style="border-bottom:1px solid #444;">'
            f'<td style="{_tag_td}{_color_tag(tag)}">{_html.escape(tag)}<br>{badge}</td>'
            f'<td style="{_row}">{short}</td></tr>'
        )

    parts.append(
        f'<div style="{_card}">'
        f'<div style="{_hdr}">'
        f'<div style="font-size:15px;font-weight:bold;margin-bottom:4px;">{_html.escape(item_id)}</div>'
        f'<div style="font-size:12px;color:#a0a0a0;">Task: {_html.escape(item["A"][:100])}</div>'
        f'<div style="font-size:12px;color:#88aaff;">Ground-truth B: {gt_b}</div>'
        f'</div>'
        f'<table style="width:100%;border-collapse:collapse;">{"".join(rows)}</table>'
        f'</div>'
    )

display(HTML("\n".join(parts)))

## 5. Generalization Test: Novel Objects

The strongest test of internalization: does the model produce mechanism-first reasoning for objects that were **never** in the ABCD dataset? If so, it has learned the *pattern*, not just memorized the training examples.

In [ ]:
"""Cell 18: Generalization test with novel PS problems."""

NOVEL_PROMPTS = [
    "Problem Solving Task\n\nProblem: How would you design a classroom that maximizes student collaboration without increasing noise levels?\n\nPropose one creative, non-obvious solution to this problem.\n- First, briefly reframe or re-analyze the problem (1-2 sentences).\n- Then describe your solution in detail (3-5 sentences).\n- Explain why it works.",
    "Problem Solving Task\n\nProblem: How could a restaurant reduce food waste by 50% without changing its menu?\n\nPropose one creative, non-obvious solution to this problem.\n- First, briefly reframe or re-analyze the problem (1-2 sentences).\n- Then describe your solution in detail (3-5 sentences).\n- Explain why it works.",
    "Problem Solving Task\n\nProblem: Design a system that helps elderly people maintain social connections without requiring them to learn new technology.\n\nPropose one creative, non-obvious solution to this problem.\n- First, briefly reframe or re-analyze the problem (1-2 sentences).\n- Then describe your solution in detail (3-5 sentences).\n- Explain why it works.",
    "Problem Solving Task\n\nProblem: How could a city encourage more people to use public libraries without spending more money?\n\nPropose one creative, non-obvious solution to this problem.\n- First, briefly reframe or re-analyze the problem (1-2 sentences).\n- Then describe your solution in detail (3-5 sentences).\n- Explain why it works.",
]

novel_results = []

for prompt_text in tqdm(NOVEL_PROMPTS, desc="Novel problems"):
    messages: list[dict[str, str]] = [
        {"role": "system", "content": ""},
        {"role": "user", "content": prompt_text},
    ]
    reply = loader.generate(
        messages, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
    )
    novel_results.append({
            "prompt": prompt_text,
            "reply": reply,
            "has_mechanism_line": bool(re.search(r"(?i)mechanism\s*:", reply)),
            "reply_len_words": len(reply.split()),
        })

novel_df = pd.DataFrame(novel_results)

In [ ]:

print("\nNovel-problem mechanism detection:")
print(novel_df["has_mechanism_line"].mean())
print()

for i, prompt_text in enumerate(NOVEL_PROMPTS):
    print("=" * 90)
    print(f"Problem {i+1}: {prompt_text}")
    print("-" * 90)
    short_prompt = prompt_text
    sub = novel_df[novel_df["prompt"] == short_prompt]
    for _, row in sub.iterrows():
        has_b = "\u2713" if row["has_mechanism_line"] else "\u2717"
        reply_short = row["reply"]
        print(f"\n  {has_b} mechanism  ({row['reply_len_words']} words)")
        print(f"  {reply_short}")
    print()


## 6. Quantitative Analysis

Compare pre-SFT vs post-SFT across multiple metrics:
1. **Mechanism presence rate:** Does the model produce a `Mechanism:` line?
2. **Reply structure:** Word count, number of list items.
3. **Semantic alignment with ground-truth B:** How close is the stated mechanism to the reference?
4. **Solution quality:** Token-level overlap between generated C and ground-truth C.

In [ ]:
"""Cell 20: Quantitative comparison."""


def extract_mechanism_text(reply: str) -> str | None:
    """Extract the mechanism line from a reply, if present."""
    m = re.search(r"(?i)mechanism\s*:\s*(.+?)(?:\n|$)", reply)
    return m.group(1).strip() if m else None


def count_list_items(reply: str) -> int:
    """Count lines starting with '- '."""
    return sum(1 for line in reply.split("\n") if line.strip().startswith("- "))


def jaccard_tokens(a: str, b: str) -> float:
    """Token-level Jaccard similarity."""
    a = a if isinstance(a, str) else ""
    b = b if isinstance(b, str) else ""
    sa = set(a.lower().split())
    sb = set(b.lower().split())
    if not sa and not sb:
        return 1.0
    return len(sa & sb) / len(sa | sb) if (sa | sb) else 0.0


# Combine all eval results
combined_df = pd.concat([baseline_df, postsft_df], ignore_index=True)

# Add metrics
combined_df["mechanism_text"] = combined_df["reply"].apply(extract_mechanism_text)
combined_df["n_list_items"] = combined_df["reply"].apply(count_list_items)

# Ground-truth C as string (PS: single string; AUT: list of strings)
def _gt_C_str(row):
    c = row["ground_truth_C"]
    return " ".join(c) if isinstance(c, list) else (c if isinstance(c, str) else "")

# Jaccard of generated solutions vs ground-truth C
combined_df["jaccard_vs_gt_C"] = combined_df.apply(
    lambda row: jaccard_tokens(row["reply"], _gt_C_str(row)), axis=1,
)

# Mechanism quality: token Jaccard between extracted mechanism line and gold B (rule)
combined_df["jaccard_vs_gt_B"] = combined_df.apply(
    lambda row: jaccard_tokens(
        row["mechanism_text"] or "",
        row["ground_truth_B"] or "",
    ),
    axis=1,
)

# Summary table
summary = combined_df.groupby("tag").agg(
    mechanism_rate=("has_mechanism_line", "mean"),
    mean_words=("reply_len_words", "mean"),
    mean_list_items=("n_list_items", "mean"),
    mean_jaccard_gt_C=("jaccard_vs_gt_C", "mean"),
    mean_jaccard_gt_B=("jaccard_vs_gt_B", "mean"),
).round(3)

print("Evaluation summary (held-out items):")
print(summary)
print()

# Per-item mechanism detection (eval_idx = one row per eval item; ids can duplicate in dataset)
pivot = combined_df.pivot_table(
    index="eval_idx", columns="tag", values="has_mechanism_line", aggfunc="first",
)
print("Mechanism detected per item (row = eval index):")
print(pivot)

In [ ]:
"""Cell 21: Visualization."""

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Plot 1: Mechanism presence rate by condition
ax = axes[0]
mech_rates = combined_df.groupby("tag")["has_mechanism_line"].mean()
colors = ["#ff6b6b" if "baseline" in t else "#51cf66" for t in mech_rates.index]
ax.bar(range(len(mech_rates)), mech_rates.values, color=colors, edgecolor="#333")
ax.set_xticks(range(len(mech_rates)))
ax.set_xticklabels(mech_rates.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Mechanism presence rate")
ax.set_title("Does the model state a mechanism?")
ax.set_ylim(0, 1.1)
ax.axhline(1.0, color="gray", linestyle="--", alpha=0.3)
ax.grid(True, alpha=0.2, axis="y")

# Plot 2: Reply length comparison
ax = axes[1]
word_counts = combined_df.groupby("tag")["reply_len_words"].agg(["mean", "std"])
ax.bar(
    range(len(word_counts)), word_counts["mean"].values,
    yerr=word_counts["std"].values, color=colors, edgecolor="#333", capsize=4,
)
ax.set_xticks(range(len(word_counts)))
ax.set_xticklabels(word_counts.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Mean reply length (words)")
ax.set_title("Reply length")
ax.grid(True, alpha=0.2, axis="y")

# Plot 3: Jaccard similarity to ground-truth C
ax = axes[2]
jac = combined_df.groupby("tag")["jaccard_vs_gt_C"].agg(["mean", "std"])
ax.bar(
    range(len(jac)), jac["mean"].values,
    yerr=jac["std"].values, color=colors, edgecolor="#333", capsize=4,
)
ax.set_xticks(range(len(jac)))
ax.set_xticklabels(jac.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Jaccard vs ground-truth C")
ax.set_title("Solution overlap with reference")
ax.set_ylim(0, max(jac["mean"].values) * 1.5 + 0.05)
ax.grid(True, alpha=0.2, axis="y")

# Plot 4: Mechanism quality — Jaccard of stated mechanism vs gold B (rule)
ax = axes[3]
jac_b = combined_df.groupby("tag")["jaccard_vs_gt_B"].agg(["mean", "std"])
ax.bar(
    range(len(jac_b)), jac_b["mean"].values,
    yerr=jac_b["std"].values, color=colors, edgecolor="#333", capsize=4,
)
ax.set_xticks(range(len(jac_b)))
ax.set_xticklabels(jac_b.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Jaccard vs gold B")
ax.set_title("Mechanism alignment with reference")
ax.set_ylim(0, min(1.0, (jac_b["mean"].max() or 0) * 1.5 + 0.05))
ax.grid(True, alpha=0.2, axis="y")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "sft_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR / 'sft_comparison.png'}")

## 8. Save Results

In [ ]:
"""Cell 25: Save all results."""

# Eval filenames: use _type3 suffix when EVAL_PROMPT_TYPE is type3_minimal (reversible experiment)
_eval_suffix = "_type3" if EVAL_PROMPT_TYPE == "type3_minimal" else ""

# Baseline results
baseline_df.to_json(
    str(OUTPUT_DIR / f"baseline_eval_results{_eval_suffix}.json"), orient="records", indent=2,
)

# Post-SFT results
postsft_df.to_json(
    str(OUTPUT_DIR / f"postsft_eval_results{_eval_suffix}.json"), orient="records", indent=2,
)

# Novel object results
novel_df.to_json(
    str(OUTPUT_DIR / "novel_object_results.json"), orient="records", indent=2,
)

# Combined summary
summary.to_csv(str(OUTPUT_DIR / "evaluation_summary.csv"))

# Training config for reproducibility (task + timestamp for LLM-as-judge)
config = {
    "task_type": TASK_TYPE.value,
    "eval_prompt_type": EVAL_PROMPT_TYPE,
    "train_prompt_type": TRAIN_PROMPT_TYPE,
    "run_timestamp": RUN_TIMESTAMP,
    "model": MODEL_NAME,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "target_modules": list(lora_config.target_modules),
    "num_train_epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "effective_batch_size": (
        training_args.per_device_train_batch_size
        * training_args.gradient_accumulation_steps
    ),
    "n_train": len(train_items),
    "n_eval": len(eval_items),
    "eval_ids": [item["id"] for item in eval_items],
    "train_ids": [item["id"] for item in train_items],
    "final_train_loss": train_result.training_loss,
    "sft_system_prompt": "",
}
with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"All outputs saved to {OUTPUT_DIR}/")
print("Files:")
for fp in sorted(OUTPUT_DIR.iterdir()):
    if fp.is_file():
        print(f"  {fp.name}  ({fp.stat().st_size / 1024:.1f} KB)")

In [ ]:
"sft_bridge_outputs/aut_20260223_054646/"